# Step 2 – Validation of the applicant name harmonisation

This notebook checks the harmonisation built in `step2Ludo_final.ipynb`.
It reads the final table of organisation names exported at the end of that notebook (`org_groups.csv`).

In [ ]:
import pandas as pd

# One row per organisation psn_id, with the original name, the cleaned name and the final group
# (exported at the end of step2Ludo_final.ipynb)
org = pd.read_csv("org_groups.csv")
org.head()

## 1. Validation sample

We validate the final grouping (`group_final`) on **our own data**, not on the course gold standard
(its names are mostly large industrial brands and say little about quantum computing applicants; see section 2).

The 128 anchors and all their candidate names were already checked by hand (three independent annotators),
so checking them again would only confirm our own labels. We therefore **exclude every name that appears in
the candidates file** (as anchor or as candidate) and draw the validation sample from the remaining names,
i.e. the part of the data harmonised only by the automatic steps (cleaning + typo merges).

In [ ]:
# Inputs
CANDIDATES_FILE = "candidates_top128_filled_brand.csv"   # hand-checked pairs (anchors + candidates)
SAMPLE_FILE     = "validation_sample.csv"                # file to label by hand
N_SAMPLE        = 100
SEED            = 42                                     # fixed seed: same sample at every run

# Names already reviewed by hand: anchors and candidates
cand = pd.read_csv(CANDIDATES_FILE)
reviewed = set(cand["anchor"].str.strip()) | set(cand["candidate"].str.strip())
print("Names already reviewed by hand:", len(reviewed))

In [ ]:
# Population: organisation names never seen in the hand review.
# We check both the cleaned name and the original name, because the candidates file
# contains cleaned names (plus a few original CJK names).
already_seen = (org["clean"].astype(str).str.strip().isin(reviewed)
                | org["psn_name"].astype(str).str.strip().isin(reviewed))
pool = org.loc[~already_seen, ["psn_name", "group_final"]].drop_duplicates("psn_name")

print("Organisation names in total:", org["psn_name"].nunique())
print("Excluded (already reviewed): ", org.loc[already_seen, "psn_name"].nunique())
print("Validation population:       ", len(pool))

**Sampling.** We draw a simple random sample of 100 names, with a fixed seed so that the sample
(and our labels) stay the same every time the notebook is run.

A simple random sample mostly contains small applicants (most names have one or two families), so it
measures the quality of the automatic steps on the tail. The head is covered by the hand review above.
Each name is judged against the whole dataset, not only against the other 99 names: a pairwise
precision/recall inside 100 random names would be almost empty, because two random tail names
almost never belong to the same organisation.

In [ ]:
import os

# Write the file to label only once, so that a re-run does not overwrite our labels
if os.path.exists(SAMPLE_FILE):
    print(SAMPLE_FILE, "already exists: not overwritten")
else:
    sample = pool.sample(n=N_SAMPLE, random_state=SEED).sort_values("psn_name")
    sample["error"] = ""          # to fill by hand: 0 = correct, 1 = wrong
    sample.to_csv(SAMPLE_FILE, index=False, encoding="utf-8-sig")
    print(SAMPLE_FILE, "written:", len(sample), "names")

**Labelling rule** (column `error`):

- **0** – `group_final` is the right organisation for this name.
- **1** – the name is in the wrong group (it was merged with a different organisation), **or** it should belong
  to another existing group but was left on its own (e.g. a subsidiary or a spelling variant of a larger group).

The rule for "same organisation" is the one used for the harmonisation: majority owner at filing date
(subsidiaries, IP companies and name changes included; joint ventures and spin-offs kept apart).

In [ ]:
# Read the labelled file and compute the error rate with a 95% Wilson interval
lab = pd.read_csv(SAMPLE_FILE, encoding="utf-8-sig")
lab["error"] = pd.to_numeric(lab["error"], errors="coerce")

missing = lab["error"].isna().sum()
if missing > 0:
    print("Still to label:", missing, "names")
else:
    n, k = len(lab), int(lab["error"].sum())
    z = 1.96
    p = k / n
    centre = (p + z**2 / (2*n)) / (1 + z**2 / n)
    half   = z * ((p*(1-p)/n + z**2 / (4*n**2)) ** 0.5) / (1 + z**2 / n)

    print(f"Wrong names: {k} / {n} = {p:.1%}")
    print(f"95% interval: {centre-half:.1%} - {centre+half:.1%}")
    display(lab[lab["error"] == 1])